# 4b. Customer concentration

**What an investor reads into this.** If a few customers carry a large share of revenue, losing one of them can undo years of growth, and those customers have bargaining power at renewal. A buyer prices that risk through the multiple, through specific protections, or by asking to see the contracts of the largest customers. The trend of the largest customers matters as much as their share: a top customer that is shrinking is revenue already leaving.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table, gbp, pct
import charts as ch

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
con = connect()
cy = sql_file(con, "customer_year")
years = sorted(cy.year.unique())
latest = max(years)

## 1. Top 1, 5, 10 and 20 shares by year, on total and on recurring revenue

In [2]:
rows = []
for y in years:
    d = cy[cy.year == y]
    tot, rec = d.net_revenue.sum(), d.recurring_net.sum()
    by_net = d.sort_values(["net_revenue", "customer_id"], ascending=[False, True]).net_revenue.values
    by_rec = d.sort_values(["recurring_net", "customer_id"], ascending=[False, True]).recurring_net.values
    for n in (1, 5, 10, 20):
        rows.append({"year": y, "top_n": n, "net_revenue_top_n": by_net[:n].sum(), "net_revenue_total": tot,
                     "share_of_net_pct": 100 * by_net[:n].sum() / tot,
                     "recurring_top_n": by_rec[:n].sum(), "recurring_total": rec,
                     "share_of_recurring_pct": 100 * by_rec[:n].sum() / rec})
topn = save_table(pd.DataFrame(rows), "4b_top_n_shares")
topn.pivot(index="top_n", columns="year", values="share_of_net_pct").round(1)

year,2022,2023,2024,2025
top_n,,,,
1,7.7,6.9,6.2,4.6
5,29.8,26.3,23.6,18.6
10,33.4,29.8,26.7,22.3
20,35.0,31.2,28.0,25.5


In [3]:
large = q(con, "SELECT customer_id FROM dim_customer WHERE is_large_account").customer_id
la = cy[cy.customer_id.isin(large)].groupby("year")[["net_revenue", "recurring_net"]].sum().reset_index()
la = la.merge(cy.groupby("year")[["net_revenue", "recurring_net"]].sum().reset_index(), on="year", suffixes=("_large", "_total"))
la["large_share_of_net_pct"] = 100 * la.net_revenue_large / la.net_revenue_total
la["large_share_of_recurring_pct"] = 100 * la.recurring_net_large / la.recurring_net_total
la["large_accounts"] = len(large)
save_table(la, "4b_large_account_share")

,year,net_revenue_large,recurring_net_large,net_revenue_total,recurring_net_total,large_share_of_net_pct,large_share_of_recurring_pct,large_accounts
0,2022,9245000.04,9245000.04,28232151.64,27248292.05,32.746353,33.928732,6
1,2023,9365000.04,9365000.04,32192869.95,31014666.15,29.090293,30.195392,6
2,2024,9722999.88,9722999.88,37258898.51,36114683.69,26.095779,26.922567,6
3,2025,9532897.62,9532897.62,45756419.35,40475097.15,20.834011,23.552501,6


## 2. Customers above 5% and 10% of net revenue in any year

In [4]:
tot = cy.groupby("year").net_revenue.transform("sum")
cy["share_pct"] = 100 * cy.net_revenue / tot
flags = cy[cy.share_pct > 5].copy()
flags["above_10pct"] = flags.share_pct > 10
flags = flags.merge(q(con, "SELECT customer_id, customer_name_std, industry, company_size FROM dim_customer"), on="customer_id")
save_table(flags[["year", "customer_id", "customer_name_std", "net_revenue", "share_pct", "above_10pct", "industry", "company_size"]]
           .sort_values(["year", "share_pct"], ascending=[True, False]), "4b_customers_above_5pct")

,year,customer_id,customer_name_std,net_revenue,share_pct,above_10pct,industry,company_size
0,2022,C1052,Brightwater Labs Group Ltd,2167499.92,7.677417,False,Healthcare,Large (250+)
1,2022,C1545,Dovedale Labs Group Ltd,1955000.04,6.924729,False,Financial Services,Large (250+)
2,2022,C1865,Kingsmead Systems & Co,1800000.00,6.375710,False,Retail,Large (250+)
3,2023,C1052,Brightwater Labs Group Ltd,2209999.92,6.864874,False,Healthcare,Large (250+)
4,2023,C1545,Dovedale Labs Group Ltd,1955000.04,6.072773,False,Financial Services,Large (250+)
5,2023,C1865,Kingsmead Systems & Co,1800000.00,5.591300,False,Retail,Large (250+)
6,2024,C1052,Brightwater Labs Group Ltd,2320499.88,6.228042,False,Healthcare,Large (250+)
7,2024,C1545,Dovedale Labs Group Ltd,2052750.00,5.509422,False,Financial Services,Large (250+)
8,2024,C1865,Kingsmead Systems & Co,1890000.00,5.072614,False,Retail,Large (250+)


## 3. Pareto curve, latest year

In [5]:
d = cy[cy.year == latest].sort_values(["net_revenue", "customer_id"], ascending=[False, True]).reset_index(drop=True)
d["rank"] = np.arange(1, len(d) + 1)
d["cum_share_pct"] = 100 * d.net_revenue.cumsum() / d.net_revenue.sum()
d["customer_pct"] = 100 * d["rank"] / len(d)
save_table(d[["rank", "customer_id", "net_revenue", "cum_share_pct", "customer_pct"]], "4b_pareto_latest_year")
pareto_marks = {p: float(d.loc[d.customer_pct >= p, "cum_share_pct"].iloc[0]) for p in (1, 10, 20, 50)}
pareto_marks

{1: 24.664595373326563,
 10: 43.59971563203188,
 20: 57.66709048661604,
 50: 82.75433036042399}

In [6]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(d.customer_pct, d.cum_share_pct, color=ch.BLUE, lw=2)
ax.plot([0, 100], [0, 100], color=ch.TEXT_2, lw=1, ls=(0, (3, 3)))
ax.text(80, 72, "equal shares", color=ch.TEXT_2, fontsize=8)
for p, pos in ((1, (6, 86)), (20, (48, 24))):
    v = pareto_marks[p]
    ax.scatter([p], [v], color=ch.BLUE, s=36, zorder=3, edgecolor=ch.SURFACE, linewidth=1.5)
    ax.annotate(f"Top {p}% of customers:\n{v:.1f}% of revenue", xy=(p, v), xytext=pos, fontsize=8, color=ch.TEXT,
                arrowprops=dict(arrowstyle="-", color=ch.TEXT_2, lw=0.8))
ax.set_xlim(0, 100); ax.set_ylim(0, 102)
ax.xaxis.set_major_formatter(ch.pct_fmt()); ax.yaxis.set_major_formatter(ch.pct_fmt())
ax.set_xlabel(f"Customers, ranked by {latest} net revenue (% of {len(d):,} customers)")
ax.set_ylabel("Cumulative share of net revenue (%)")
ax.set_title(f"Revenue concentration, {latest} (Pareto curve)")
print(ch.save(fig, "4b_pareto"))

outputs/charts/4b_pareto.png


## 4. Top 10 customers: quarterly net revenue trend (ranked on the latest year)

In [7]:
top10 = d.customer_id.head(10).tolist()
qtr = q(con, f"""
    SELECT customer_id, year(month_start) || 'Q' || quarter(month_start) AS quarter,
           sum(net_revenue) AS net_revenue, sum(recurring_net) AS recurring_net, sum(oneoff_net) AS oneoff_net
    FROM fact_revenue_monthly WHERE customer_id IN ({','.join(repr(c) for c in top10)})
    GROUP BY ALL ORDER BY customer_id, quarter
""")
grid = pd.MultiIndex.from_product([top10, [f"{y}Q{qq}" for y in years for qq in range(1, 5)]], names=["customer_id", "quarter"])
qtr = qtr.set_index(["customer_id", "quarter"]).reindex(grid, fill_value=0).reset_index()
qtr["rank_latest_year"] = qtr.customer_id.map({c: i + 1 for i, c in enumerate(top10)})
save_table(qtr, "4b_top10_quarterly")

,customer_id,quarter,net_revenue,recurring_net,oneoff_net,rank_latest_year
0,C1052,2022Q1,509999.98,509999.98,0.0,1
1,C1052,2022Q2,552499.98,552499.98,0.0,1
2,C1052,2022Q3,552499.98,552499.98,0.0,1
3,C1052,2022Q4,552499.98,552499.98,0.0,1
4,C1052,2023Q1,552499.98,552499.98,0.0,1
...,...,...,...,...,...,...
155,C1964,2024Q4,5890.50,5890.50,0.0,10
156,C1964,2025Q1,6302.85,6302.85,0.0,10
157,C1964,2025Q2,6302.85,6302.85,0.0,10
158,C1964,2025Q3,6302.85,6302.85,0.0,10


In [8]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5.2), sharex=True)
quarters = qtr.quarter.unique()
for ax, cid in zip(axes.flat, top10):
    s = qtr[qtr.customer_id == cid]
    ax.bar(range(len(quarters)), s.recurring_net, color=ch.BLUE, width=0.8, label="Recurring")
    ax.bar(range(len(quarters)), s.oneoff_net, bottom=s.recurring_net, color=ch.ORANGE, width=0.8, label="One-off")
    r = int(s.rank_latest_year.iloc[0])
    ax.set_title(f"#{r} {cid}", fontsize=9)
    ax.yaxis.set_major_formatter(ch.gbp_k())
    ax.set_xticks([0, 4, 8, 12]); ax.set_xticklabels(["2022", "2023", "2024", "2025"], fontsize=7)
    ax.tick_params(axis="y", labelsize=7)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=2, fontsize=8)
fig.suptitle(f"Top 10 customers by {latest} net revenue: quarterly net revenue (£k, each panel on its own scale)",
             x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout(rect=(0, 0.03, 1, 0.95))
print(ch.save(fig, "4b_top10_trend"))

outputs/charts/4b_top10_trend.png


## 5. Declining top customers: run-rate impact

In [9]:
mrr_c = q(con, "SELECT customer_id, month_start, sum(mrr) AS mrr FROM fact_mrr_monthly GROUP BY ALL")
rows = []
for cid in top10:
    m = mrr_c[mrr_c.customer_id == cid].set_index("month_start").mrr.sort_index()
    dec = lambda y: 12 * float(m.get(pd.Timestamp(f"{y}-12-01"), 0.0))
    peak_month = m.idxmax() if len(m) else None
    rows.append({"customer_id": cid, "rank_latest_year": top10.index(cid) + 1,
                 "dec_arr_prior": dec(latest - 1), "dec_arr_latest": dec(latest),
                 "change_vs_prior_dec": dec(latest) - dec(latest - 1),
                 "change_vs_prior_dec_pct": 100 * (dec(latest) / dec(latest - 1) - 1) if dec(latest - 1) else np.nan,
                 "peak_arr": 12 * float(m.max()) if len(m) else 0.0, "peak_month": peak_month,
                 "change_vs_peak": dec(latest) - 12 * float(m.max()) if len(m) else 0.0})
runrate = pd.DataFrame(rows)
runrate["declining"] = runrate.change_vs_prior_dec < 0
runrate["change_vs_peak_pct"] = 100 * runrate.change_vs_peak / runrate.peak_arr
save_table(runrate, "4b_top10_runrate")
runrate

,customer_id,rank_latest_year,dec_arr_prior,dec_arr_latest,change_vs_prior_dec,change_vs_prior_dec_pct,peak_arr,peak_month,change_vs_peak,declining,change_vs_peak_pct
0,C1052,1,2320499.88,1336964.88,-983535.00,-42.384618,2482934.88,2025-01-01,-1145970.00,True,-46.153848
1,C1865,2,1890000.00,2022300.00,132300.00,7.000000,2022300.00,2025-01-01,0.00,False,0.000000
2,C1545,3,2052750.00,1808835.00,-243915.00,-11.882353,2196442.56,2025-01-01,-387607.56,True,-17.647061
3,C1881,4,1249500.00,1336965.00,87465.00,7.000000,1470000.00,2024-01-01,-133035.00,False,-9.050000
4,C1328,5,1155000.00,1235850.12,80850.12,7.000010,1235850.12,2025-01-01,0.00,False,0.000000
5,C1933,6,945000.00,1011150.12,66150.12,7.000013,1011150.12,2025-01-01,0.00,False,0.000000
6,C1069,7,42607.20,45589.68,2982.48,6.999944,45589.68,2025-01-01,0.00,False,0.000000
7,C2470,8,42053.64,44997.36,2943.72,6.999917,44997.36,2025-01-01,0.00,False,0.000000
8,C2321,9,42140.52,45090.36,2949.84,7.000009,45090.36,2025-01-01,0.00,False,0.000000
9,C1964,10,23562.00,25211.40,1649.40,7.000255,25211.40,2025-01-01,0.00,False,0.000000


In [10]:
# What was lost: product lines of the declining customers, December ARR by line
decl = runrate[runrate.declining].customer_id.tolist()
lines = q(con, f"""
    SELECT m.customer_id, m.product_line, year(m.month_start) AS year, 12 * sum(m.mrr) AS dec_arr
    FROM fact_mrr_monthly m WHERE month(m.month_start) = 12 AND m.customer_id IN ({','.join(repr(c) for c in decl) or "''"})
    GROUP BY ALL
""").pivot_table(index=["customer_id", "product_line"], columns="year", values="dec_arr", fill_value=0).reset_index()
ends = q(con, f"""
    SELECT s.customer_id, p.product_line, s.end_date FROM clean_subscriptions s JOIN clean_products p USING (product_id)
    WHERE s.end_date >= DATE '2022-01-01' AND s.customer_id IN ({','.join(repr(c) for c in decl) or "''"})
""")
lines = lines.merge(ends, on=["customer_id", "product_line"], how="left")
lines.columns = [str(c) for c in lines.columns]
save_table(lines, "4b_declining_customer_lines")
lines

,customer_id,product_line,2022,2023,2024,2025,end_date
0,C1052,Analytics Add-on,510000.00,510000.00,535500.00,0.00,2025-09-01
1,C1052,Core Platform,1189999.92,1189999.92,1249499.88,1336964.88,NaT
2,C1052,Payments Module,510000.00,510000.00,535500.00,0.00,2025-09-01
3,C1545,Core Platform,1610000.04,1610000.04,1690500.00,1808835.00,NaT
4,C1545,Payments Module,345000.00,345000.00,362250.00,0.00,2025-02-01


In [11]:
display(topn); display(la); flags

,year,top_n,net_revenue_top_n,net_revenue_total,share_of_net_pct,recurring_top_n,recurring_total,share_of_recurring_pct
0,2022,1,2167499.92,28232151.64,7.677417,2167499.92,27248292.05,7.954627
1,2022,5,8408750.04,28232151.64,29.784305,8408750.04,27248292.05,30.859733
2,2022,10,9436266.60,28232151.64,33.423831,9436266.60,27248292.05,34.630672
3,2022,20,9891498.74,28232151.64,35.036291,9891498.74,27248292.05,36.301353
4,2023,1,2209999.92,32192869.95,6.864874,2209999.92,31014666.15,7.125661
5,2023,5,8465000.04,32192869.95,26.294642,8465000.04,31014666.15,27.293539
6,2023,10,9580880.47,32192869.95,29.760877,9561310.09,31014666.15,30.828351
7,2023,20,10048874.89,32192869.95,31.214598,10020724.77,31014666.15,32.309633
8,2024,1,2320499.88,37258898.51,6.228042,2320499.88,36114683.69,6.425364
9,2024,5,8777999.88,37258898.51,23.559472,8777999.88,36114683.69,24.305903


,year,net_revenue_large,recurring_net_large,net_revenue_total,recurring_net_total,large_share_of_net_pct,large_share_of_recurring_pct,large_accounts
0,2022,9245000.04,9245000.04,28232151.64,27248292.05,32.746353,33.928732,6
1,2023,9365000.04,9365000.04,32192869.95,31014666.15,29.090293,30.195392,6
2,2024,9722999.88,9722999.88,37258898.51,36114683.69,26.095779,26.922567,6
3,2025,9532897.62,9532897.62,45756419.35,40475097.15,20.834011,23.552501,6


,customer_id,year,net_revenue,recurring_net,other_implementation,dec_arr,share_pct,above_10pct,customer_name_std,industry,company_size
0,C1052,2022,2167499.92,2167499.92,0.0,2209999.92,7.677417,False,Brightwater Labs Group Ltd,Healthcare,Large (250+)
1,C1545,2022,1955000.04,1955000.04,0.0,1955000.04,6.924729,False,Dovedale Labs Group Ltd,Financial Services,Large (250+)
2,C1865,2022,1800000.00,1800000.00,0.0,1800000.00,6.375710,False,Kingsmead Systems & Co,Retail,Large (250+)
3,C1052,2023,2209999.92,2209999.92,0.0,2209999.92,6.864874,False,Brightwater Labs Group Ltd,Healthcare,Large (250+)
4,C1545,2023,1955000.04,1955000.04,0.0,1955000.04,6.072773,False,Dovedale Labs Group Ltd,Financial Services,Large (250+)
5,C1865,2023,1800000.00,1800000.00,0.0,1800000.00,5.591300,False,Kingsmead Systems & Co,Retail,Large (250+)
6,C1052,2024,2320499.88,2320499.88,0.0,2320499.88,6.228042,False,Brightwater Labs Group Ltd,Healthcare,Large (250+)
7,C1545,2024,2052750.00,2052750.00,0.0,2052750.00,5.509422,False,Dovedale Labs Group Ltd,Financial Services,Large (250+)
8,C1865,2024,1890000.00,1890000.00,0.0,1890000.00,5.072614,False,Kingsmead Systems & Co,Retail,Large (250+)
